# 05 · ONNX export for the in-browser demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rishabh-bgp/brain-tumor-mri-classification/blob/main/notebooks/05_onnx_export_and_web_demo.ipynb)

Exports the leakage-aware model to ONNX with two outputs, the class logits and the class activation maps, and checks the export against PyTorch. For a global-average-pooled linear head, Grad-CAM equals CAM up to a positive scale, so the browser can compute exact Grad-CAM without gradients.

*Rishabh Aryan · M.Tech (AI & Data Science), IIIT Bhagalpur. Research and education code, not a medical device.*

In [1]:
# --- Setup: works locally (repo checkout) and on Google Colab ---
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("brain-tumor-mri-classification"):
        subprocess.run(["git", "clone", "-q", "https://github.com/Rishabh-bgp/brain-tumor-mri-classification.git"], check=True)
    os.chdir("brain-tumor-mri-classification")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "kagglehub", "imagehash", "captum",
                    "onnx", "onnxruntime", "scikit-image"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("working directory:", os.getcwd())

working directory: /workspace/brain-tumor-mri/repo


In [2]:
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU (Colab: Runtime > Change runtime type > GPU for faster training)")

PyTorch 2.15.0+cpu | CUDA available: False
Running on CPU (Colab: Runtime > Change runtime type > GPU for faster training)


In [3]:
subprocess.run([sys.executable, "scripts/export_onnx.py"], check=True)
import json; print(json.dumps(json.load(open("results/onnx_verification.json")), indent=1))

/workspace/brain-tumor-mri/repo/scripts/export_onnx.py:23: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(wrap, torch.zeros(1, 1, 224, 224), str(onnx_path), input_names=["image"], output_names=["logits", "cam"],


{
  "max_abs_logit_diff_torch_vs_onnx": 0.000354766845703125,
  "argmax_agreement": 1.0,
  "max_abs_cam_diff_torch_vs_onnx": 0.00157928466796875,
  "max_abs_diff_onnx_cam_vs_autograd_gradcam_normalised": 7.2904549351937575e-06,
  "onnx_size_mb": 16.027229
}


{
 "max_abs_logit_diff_torch_vs_onnx": 0.000354766845703125,
 "argmax_agreement": 1.0,
 "max_abs_cam_diff_torch_vs_onnx": 0.00157928466796875,
 "max_abs_diff_onnx_cam_vs_autograd_gradcam_normalised": 7.2904549351937575e-06,
 "onnx_size_mb": 16.027229
}


In [4]:
import onnxruntime as ort, numpy as np, pandas as pd
from btmri.data import download_dataset, load_gray_square
root = download_dataset()
sess = ort.InferenceSession("docs/model/brain_tumor_effnetb0.onnx")
print([(i.name, i.shape) for i in sess.get_inputs()], [(o.name, o.shape) for o in sess.get_outputs()])
pred = pd.read_csv("results/predictions/clean_test_predictions.csv").sample(5, random_state=0)
for _, r in pred.iterrows():
    x = load_gray_square(str(root / r.relpath), 224).astype(np.float32)[None, None] / 255
    logits, cam = sess.run(None, {"image": x})
    print(r.relpath, "| true", r.true, "| ONNX argmax", ["glioma", "meningioma", "notumor", "pituitary"][logits.argmax()], "| saved pred", r.pred)

[('image', ['batch', 1, 224, 224])] [('logits', ['batch', 4]), ('cam', ['batch', 4, 7, 7])]
Testing/notumor/Te-no_345.jpg | true notumor | ONNX argmax notumor | saved pred notumor
Testing/glioma/Te-gl_393.jpg | true glioma | ONNX argmax glioma | saved pred glioma
Testing/pituitary/Te-pi_60.jpg | true pituitary | ONNX argmax pituitary | saved pred pituitary
Testing/notumor/Te-no_205.jpg | true notumor | ONNX argmax notumor | saved pred notumor
Testing/notumor/Te-no_256.jpg | true notumor | ONNX argmax notumor | saved pred notumor


Serve the demo locally with `cd docs && python -m http.server 8000`, or open the live site: https://rishabh-bgp.github.io/brain-tumor-mri-classification/